# Sigma Sweep: Tightness vs Privacy Regime

**EECE 608 -- Mohamad Faour**

This notebook sweeps across noise multipliers (sigma) to measure how the
tightness gap changes across privacy regimes. For each sigma:
1. Train a target model with DP-SGD at that sigma
2. Compute epsilon_upper via both RDP and PLD accounting
3. Train K shadow models and run Raw LiRA
4. Compute epsilon_lower (Wilson CI conservative)
5. Decompose the gap: accounting gap (RDP vs PLD) + auditing gap

**Key question**: Does tightness improve at higher epsilon (weaker privacy)?

## 0. Setup

In [ ]:
import os, sys, shutil
from pathlib import Path

WORK = Path("/content/EECE_608")
if WORK.exists():
    shutil.rmtree(WORK)
!git clone https://github.com/mohdfaour03/EECE_608.git /content/EECE_608

os.chdir(WORK)
!pip install -q opacus>=1.5 dp-accounting>=0.4 pyyaml
!pip install -e . -q

sys.path.insert(0, str(WORK / "src"))
sys.path.insert(0, str(WORK / "autoresearch"))

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print("Ready.")

## 1. Imports and helpers

In [ ]:
import prepare
import torch
import torch.nn.functional as F
import random
import time
import json
import logging
import statistics
import numpy as np
from pathlib import Path
from torch.utils.data import Subset

from dp_audit_tightness.privacy.empirical import estimate_empirical_lower_bound
from dp_audit_tightness.privacy.pld_accounting import compute_epsilon_pld
from dp_audit_tightness.training.dp_sgd import train_dp_sgd
from dp_audit_tightness.data.datasets import load_dataset_bundle, DatasetBundle
from dp_audit_tightness.utils.seeds import set_global_seed
from dp_audit_tightness.models.io import load_model_for_inference

logging.basicConfig(level=logging.WARNING, format='%(message)s')
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load dataset once (shared across all sigma values)
bundle = load_dataset_bundle(
    prepare._build_train_config().dataset,
    split_seed=prepare.SPLIT_SEED,
)
n_train = len(bundle.train_dataset)
print(f"Dataset: {n_train} train, {len(bundle.eval_dataset)} eval")
print(f"Device: {device}")

In [ ]:
def train_model(noise_multiplier, seed, dataset=None):
    """Train a DP-SGD model with given noise_multiplier. Returns (model, record)."""
    config = prepare._build_train_config(
        noise_multiplier=noise_multiplier, seed=seed
    )
    set_global_seed(seed)
    logger = logging.getLogger(f"train_s{seed}_nm{noise_multiplier}")
    logger.setLevel(logging.WARNING)

    if dataset is not None:
        ds_bundle = DatasetBundle(
            train_dataset=dataset,
            eval_dataset=bundle.eval_dataset,
            input_dim=prepare.INPUT_DIM,
            num_classes=prepare.NUM_CLASSES,
            train_size=len(dataset),
            eval_size=len(bundle.eval_dataset),
        )
        outcome = train_dp_sgd(config=config, logger=logger, dataset_bundle=ds_bundle)
    else:
        outcome = train_dp_sgd(config=config, logger=logger)

    model = load_model_for_inference(config.model, outcome.checkpoint_path, device=device)
    return model, outcome.record


def compute_losses(mdl, dataset, indices, batch_size=512):
    """Compute per-example CE loss."""
    all_losses = []
    with torch.no_grad():
        for s in range(0, len(indices), batch_size):
            bi = indices[s:s+batch_size]
            imgs, lbls = zip(*(dataset[i] for i in bi))
            x = torch.stack(imgs).to(device)
            y = torch.tensor(lbls, dtype=torch.long, device=device)
            losses = F.cross_entropy(mdl(x), y, reduction="none")
            all_losses.append(losses.cpu())
    return torch.cat(all_losses).numpy()


def run_lira_audit(target_model, noise_multiplier, K=32, budget=256, seeds=[401,402,403,404,405]):
    """Train K shadows, run Raw LiRA, return (member_scores, nonmember_scores)."""
    half = n_train // 2

    # Train shadow models
    shadow_models = []
    shadow_member_sets = []
    for k in range(K):
        shadow_seed = 5000 + k
        rng_k = random.Random(shadow_seed)
        all_indices = list(range(n_train))
        rng_k.shuffle(all_indices)
        in_indices = sorted(all_indices[:half])
        shadow_member_sets.append(set(in_indices))

        subset = Subset(bundle.train_dataset, in_indices)
        sm, _ = train_model(noise_multiplier, shadow_seed, dataset=subset)
        shadow_models.append(sm)

    # Collect query indices across seeds
    max_budget = max(budget, 256)
    all_train_idx = set()
    all_eval_idx = set()
    for seed in seeds:
        rng = random.Random(seed)
        all_train_idx.update(rng.sample(range(n_train), max_budget // 2))
        all_eval_idx.update(rng.sample(range(len(bundle.eval_dataset)), max_budget // 2))
    all_train_idx = sorted(all_train_idx)
    all_eval_idx = sorted(all_eval_idx)
    train_idx_to_pos = {idx: pos for pos, idx in enumerate(all_train_idx)}
    eval_idx_to_pos = {idx: pos for pos, idx in enumerate(all_eval_idx)}

    # Precompute loss matrices
    loss_matrix_train = np.zeros((K, len(all_train_idx)), dtype=np.float32)
    loss_matrix_eval = np.zeros((K, len(all_eval_idx)), dtype=np.float32)
    for k, sm in enumerate(shadow_models):
        loss_matrix_train[k] = compute_losses(sm, bundle.train_dataset, all_train_idx)
        loss_matrix_eval[k] = compute_losses(sm, bundle.eval_dataset, all_eval_idx)

    # Raw LiRA scoring
    all_m, all_n = [], []
    for seed in seeds:
        rng = random.Random(seed)
        mi = rng.sample(list(train_idx_to_pos.keys()), budget // 2)
        ni = rng.sample(list(eval_idx_to_pos.keys()), budget // 2)

        # Member scores
        for idx in mi:
            pos = train_idx_to_pos[idx]
            in_losses, out_losses = [], []
            for k in range(K):
                lk = float(loss_matrix_train[k, pos])
                if idx in shadow_member_sets[k]:
                    in_losses.append(lk)
                else:
                    out_losses.append(lk)
            if in_losses and out_losses:
                all_m.append(np.mean(out_losses) - np.mean(in_losses))
            elif out_losses:
                all_m.append(np.mean(out_losses))
            else:
                all_m.append(0.0)

        # Non-member scores
        target_loss_eval = compute_losses(target_model, bundle.eval_dataset, ni)
        for j, idx in enumerate(ni):
            pos = eval_idx_to_pos[idx]
            shadow_losses = [float(loss_matrix_eval[k, pos]) for k in range(K)]
            all_n.append(np.mean(shadow_losses) - float(target_loss_eval[j]))

    return all_m, all_n


print("Helpers defined.")

## 2. Sigma sweep

For each noise multiplier, we:
1. Train a target model
2. Compute epsilon_upper (RDP + PLD)
3. Train K=32 shadow models + run Raw LiRA (budget=256, 5 seeds)
4. Compute epsilon_lower (Wilson CI conservative)

Expected time: ~4 min per sigma (1 target + 32 shadows @ ~7s each)

In [ ]:
SIGMAS = [0.5, 0.8, 1.1, 1.5, 2.0, 3.0, 5.0]
K = 32
BUDGET = 256
SEEDS = [401, 402, 403, 404, 405]

sweep_results = []

print("=" * 100)
print(f"SIGMA SWEEP: K={K} shadows, budget={BUDGET}, {len(SEEDS)} seeds")
print("=" * 100)

for sigma in SIGMAS:
    t0 = time.time()
    print(f"\n--- sigma = {sigma} ---")

    # 1. Train target model
    target_model, record = train_model(sigma, seed=prepare.TRAINING_SEED)
    eps_rdp = record.epsilon_upper_theory
    eps_pld = record.epsilon_upper_pld
    accuracy = record.utility_metrics.get("accuracy", 0)

    # If PLD not available, try computing it
    if eps_pld is None:
        try:
            sampling_rate = prepare.BATCH_SIZE / n_train
            num_steps = prepare.EPOCHS * (n_train // prepare.BATCH_SIZE)
            pld_result = compute_epsilon_pld(
                noise_multiplier=sigma,
                sampling_rate=sampling_rate,
                num_steps=num_steps,
                delta=prepare.DELTA,
            )
            eps_pld = pld_result["epsilon_pld"]
        except Exception as e:
            print(f"  PLD failed: {e}")
            eps_pld = eps_rdp  # fallback

    print(f"  eps_rdp={eps_rdp:.4f}  eps_pld={eps_pld:.4f}  accuracy={accuracy:.4f}")

    # 2. Run LiRA audit
    member_scores, nonmember_scores = run_lira_audit(
        target_model, noise_multiplier=sigma, K=K, budget=BUDGET, seeds=SEEDS
    )

    # 3. Compute epsilon_lower (Wilson CI conservative)
    est_cons = estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=prepare.DELTA,
        align_event_to_score_direction=True,
        require_member_favoring=True,
        report_confidence_supported_lower_bound=True,
    )
    est_point = estimate_empirical_lower_bound(
        member_scores=member_scores,
        nonmember_scores=nonmember_scores,
        delta=prepare.DELTA,
        align_event_to_score_direction=True,
        require_member_favoring=True,
        report_confidence_supported_lower_bound=False,
    )

    eps_lower_cons = est_cons.epsilon_lower_empirical
    eps_lower_point = est_point.epsilon_lower_empirical

    # 4. Gap decomposition
    accounting_gap = eps_rdp - eps_pld  # how much tighter PLD is vs RDP
    total_gap = eps_rdp - eps_lower_cons
    auditing_gap = eps_pld - eps_lower_cons  # gap remaining after tighter accounting

    # Tightness ratios (using best available upper bound = PLD)
    tr_cons_rdp = eps_lower_cons / eps_rdp if eps_rdp > 0 else 0
    tr_cons_pld = eps_lower_cons / eps_pld if eps_pld > 0 else 0
    tr_point_rdp = eps_lower_point / eps_rdp if eps_rdp > 0 else 0

    elapsed = time.time() - t0
    score_gap = statistics.fmean(member_scores) - statistics.fmean(nonmember_scores)

    row = {
        "sigma": sigma,
        "eps_rdp": eps_rdp,
        "eps_pld": eps_pld,
        "eps_lower_cons": eps_lower_cons,
        "eps_lower_point": eps_lower_point,
        "accounting_gap": accounting_gap,
        "auditing_gap": auditing_gap,
        "total_gap": total_gap,
        "tr_cons_rdp": tr_cons_rdp,
        "tr_cons_pld": tr_cons_pld,
        "tr_point_rdp": tr_point_rdp,
        "accuracy": accuracy,
        "score_gap": score_gap,
        "tpr": est_cons.selected_tpr,
        "fpr": est_cons.selected_fpr,
        "n_member": len(member_scores),
        "n_nonmember": len(nonmember_scores),
        "time_s": elapsed,
    }
    sweep_results.append(row)

    print(f"  eps_lower: cons={eps_lower_cons:.4f} point={eps_lower_point:.4f}")
    print(f"  tightness (vs RDP): cons={tr_cons_rdp:.4%} point={tr_point_rdp:.4%}")
    print(f"  tightness (vs PLD): cons={tr_cons_pld:.4%}")
    print(f"  gap decomposition: accounting={accounting_gap:.4f} auditing={auditing_gap:.4f} total={total_gap:.4f}")
    print(f"  score_gap={score_gap:.4f}  time={elapsed:.0f}s")

print("\n" + "=" * 100)
print("Sweep complete.")

## 3. Results table

In [ ]:
import pandas as pd

df = pd.DataFrame(sweep_results)

print("=" * 120)
print("SIGMA SWEEP RESULTS")
print("=" * 120)
print(df[["sigma", "eps_rdp", "eps_pld", "eps_lower_cons", "eps_lower_point",
          "tr_cons_rdp", "tr_cons_pld", "tr_point_rdp",
          "accounting_gap", "auditing_gap", "accuracy"]].to_string(index=False))

print()
print("-" * 80)
print("GAP DECOMPOSITION (as % of total gap):")
for _, row in df.iterrows():
    total = row["total_gap"]
    if total > 0:
        acct_pct = row["accounting_gap"] / total * 100
        audit_pct = row["auditing_gap"] / total * 100
    else:
        acct_pct = audit_pct = 0
    print(f"  sigma={row['sigma']:.1f}: total_gap={total:.4f}  "
          f"accounting={acct_pct:.1f}%  auditing={audit_pct:.1f}%")

# Save CSV
out_dir = WORK / "autoresearch" / "results"
out_dir.mkdir(parents=True, exist_ok=True)
csv_path = out_dir / "sigma_sweep_results.csv"
df.to_csv(csv_path, index=False)
print(f"\nSaved to {csv_path}")

## 4. Plots

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Epsilon upper vs lower across sigma
ax = axes[0]
ax.plot(df["sigma"], df["eps_rdp"], "o-", label="eps_upper (RDP)", color="red")
ax.plot(df["sigma"], df["eps_pld"], "s-", label="eps_upper (PLD)", color="orange")
ax.plot(df["sigma"], df["eps_lower_cons"], "^-", label="eps_lower (conservative)", color="blue")
ax.plot(df["sigma"], df["eps_lower_point"], "v--", label="eps_lower (point)", color="lightblue")
ax.set_xlabel("Noise Multiplier (sigma)")
ax.set_ylabel("Epsilon")
ax.set_title("Privacy Bounds vs Noise Level")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 2: Tightness ratio across sigma
ax = axes[1]
ax.plot(df["sigma"], df["tr_cons_rdp"] * 100, "o-", label="vs RDP (conservative)", color="blue")
ax.plot(df["sigma"], df["tr_cons_pld"] * 100, "s-", label="vs PLD (conservative)", color="green")
ax.plot(df["sigma"], df["tr_point_rdp"] * 100, "^--", label="vs RDP (point)", color="lightblue")
ax.set_xlabel("Noise Multiplier (sigma)")
ax.set_ylabel("Tightness Ratio (%)")
ax.set_title("Tightness Ratio vs Noise Level")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# Plot 3: Gap decomposition (stacked)
ax = axes[2]
ax.bar(range(len(df)), df["accounting_gap"], label="Accounting gap (RDP - PLD)", color="orange")
ax.bar(range(len(df)), df["auditing_gap"], bottom=df["accounting_gap"], label="Auditing gap (PLD - eps_lower)", color="steelblue")
ax.set_xticks(range(len(df)))
ax.set_xticklabels([f"{s:.1f}" for s in df["sigma"]])
ax.set_xlabel("Noise Multiplier (sigma)")
ax.set_ylabel("Gap (epsilon units)")
ax.set_title("Gap Decomposition")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
fig_path = WORK / "autoresearch" / "results" / "sigma_sweep_plots.png"
plt.savefig(fig_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {fig_path}")

## 5. Accuracy vs privacy trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df["eps_rdp"], df["accuracy"] * 100, "o-", color="red", label="Accuracy vs eps_upper (RDP)")

# Annotate each point with sigma
for _, row in df.iterrows():
    ax.annotate(f"  sigma={row['sigma']}", (row["eps_rdp"], row["accuracy"] * 100), fontsize=8)

ax.set_xlabel("Epsilon (RDP upper bound)")
ax.set_ylabel("Test Accuracy (%)")
ax.set_title("Utility-Privacy Trade-off")
ax.grid(True, alpha=0.3)
ax.legend()

plt.tight_layout()
fig_path2 = WORK / "autoresearch" / "results" / "utility_privacy_tradeoff.png"
plt.savefig(fig_path2, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved to {fig_path2}")

## 6. Download results

In [ ]:
from google.colab import files

files.download(str(csv_path))
files.download(str(fig_path))
files.download(str(fig_path2))
print("Downloads triggered.")